<a href="https://colab.research.google.com/github/paolobalasso/DataScienceCourse/blob/main/notebooks/05_conjoint_demand_revenue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 05 — Conjoint Analysis → Potential Demand → Revenue

**End-to-end pipeline** in one notebook:
1. **Design** a choice-based conjoint study (attributes, levels, choice sets).
2. **Simulate respondents** with heterogeneous part-worths (so the data behaves like a real CBC survey).
3. **Estimate** an aggregate **multinomial logit (MNL)** to recover the part-worths.
4. Derive **attribute importance** and **willingness-to-pay (WTP)**.
5. Build a **market simulator**: share of preference for any product configuration.
6. Translate share → **potential demand** (households) given a market size.
7. Compute **revenue**, find the **revenue-maximising price**, and check how much that answer depends on the sample (bootstrap).

*Didactic notebook — synthetic data, about 30 s on a laptop; allow up to a minute on a free Colab CPU. The math is real, the numbers are made up. Use **Runtime → Run all**; re-running a single cell is also safe, because every random cell has its own seed.*

## 0. Setup

In [ ]:
# Everything used here ships with Google Colab and Anaconda: no installs needed.
import numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt, scipy
from itertools import product
from scipy.optimize import minimize
from scipy.stats import norm

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
print('numpy', np.__version__, '| pandas', pd.__version__,
      '| scipy', scipy.__version__, '| matplotlib', matplotlib.__version__)

## 1. Conjoint design

We are pricing a **streaming subscription**. Four attributes:

| Attribute | Levels |
|---|---|
| `Price` (EUR/month) | 7.99, 11.99, 15.99 |
| `Catalog` | Basic, Standard, Premium |
| `Ads` | With ads, No ads |
| `Devices` | 1 device, 2 devices, 4 devices |

Full factorial = 3·3·2·3 = **54 profiles**. Each respondent sees **8 choice tasks**, each task offering **3 alternatives + a 'None'** option.

The three price levels define the **tested range, 7.99 to 15.99**. Everything we say about price later stays inside it.

The code cell below also writes the **model specification once** (`design_matrix` and `utility`). Every later step reuses it, so a change to the attributes is a change in one place.

In [ ]:
PRICE   = [7.99, 11.99, 15.99]
CATALOG = ['Basic', 'Standard', 'Premium']
ADS     = ['With ads', 'No ads']
DEVICES = [1, 2, 4]

profiles = pd.DataFrame(list(product(PRICE, CATALOG, ADS, DEVICES)),
                        columns=['price','catalog','ads','devices'])
print(f'Full factorial: {len(profiles)} profiles')

# The model specification, written ONCE and reused everywhere below.
# Base levels (utility 0): Basic catalog, With ads, 1 device.
FEATURES = ['price', 'cat_Standard', 'cat_Premium', 'ads_No', 'dev_2', 'dev_4', 'none']

def design_matrix(df):
    # one row per alternative -> dummy-coded attributes (the X of the utility model)
    return pd.DataFrame({
        'price':        df['price'],
        'cat_Standard': (df['catalog'] == 'Standard').astype(int),
        'cat_Premium':  (df['catalog'] == 'Premium').astype(int),
        'ads_No':       (df['ads'] == 'No ads').astype(int),
        'dev_2':        (df['devices'] == 2).astype(int),
        'dev_4':        (df['devices'] == 4).astype(int),
        'none':         (df['catalog'] == 'NONE').astype(int),
    }, index=df.index)[FEATURES]

def utility(df, beta):
    # deterministic utility U = X @ beta for every row of df (beta: dict or Series)
    return design_matrix(df).to_numpy(float) @ pd.Series(beta)[FEATURES].to_numpy(float)

NONE_OPTION = pd.DataFrame([{'price': 0.0, 'catalog': 'NONE', 'ads': 'NONE', 'devices': 0}])
profiles.head()

## 2. Simulate 400 respondents with heterogeneous part-worths

True (population-mean) utilities — we will try to recover these in step 3:

- price: **-0.25 per EUR** (linear)
- catalog: Basic=0 (base), Standard=+0.6, Premium=+1.1
- ads: With ads=0 (base), No ads=+0.8
- devices: 1=0, 2=+0.4, 4=+0.7
- 'None' option intercept: **-2.0**

Each respondent gets these means + Gaussian noise → realistic individual heterogeneity:
- **tastes** (catalog, ads, devices, None) vary a lot: standard deviation **0.25**;
- **price sensitivity** varies a little: standard deviation **0.05** around -0.25. Everybody dislikes paying more, some people more than others.

Why not 0.25 on price too? Because then the spread would be as large as the mean (-0.25 ± 0.25), and about one respondent in six would *prefer* higher prices. That is not a market we want to teach on. Mini-exercise 5 lets you turn it on and see what happens to the estimates.

How a choice is made: each option's utility + random **Gumbel** noise, and the respondent picks the maximum. Gumbel noise + "pick the max" is exactly what makes choice probabilities follow the softmax on the slide *Choice-Based Conjoint: The Gold Standard*.

In [ ]:
N_RESP  = 400
N_TASKS = 8
N_ALTS  = 3   # plus 'None'

# population-mean ("true") part-worths: what the MNL should recover
TRUE = {'price': -0.25,
        'cat_Standard': 0.6, 'cat_Premium': 1.1,
        'ads_No': 0.8,
        'dev_2': 0.4, 'dev_4': 0.7,
        'none': -2.0}

# individual part-worths = TRUE + Gaussian noise
PRICE_SD = 0.05   # price sensitivity: small spread, nobody likes higher prices
ATTR_SD  = 0.25   # tastes for catalog, ads, devices and None: larger spread

rng_sim = np.random.default_rng(7)
mean = pd.Series(TRUE)[FEATURES].to_numpy()
sd   = np.array([PRICE_SD if f == 'price' else ATTR_SD for f in FEATURES])
betas_ind = pd.DataFrame(rng_sim.normal(mean, sd, size=(N_RESP, len(FEATURES))), columns=FEATURES)

print(f"Respondents who like higher prices (beta_price > 0): {(betas_ind['price'] > 0).mean():.1%}")
betas_ind.describe().loc[['mean', 'std', 'min', 'max']].round(2)

In [ ]:
# The CBC survey: each respondent answers N_TASKS choice tasks (3 random profiles + None).
rng_tasks = np.random.default_rng(8)
alts = pd.concat([profiles, NONE_OPTION], ignore_index=True)    # 54 profiles + None = 55 options
NONE_IDX = len(profiles)
U_all = design_matrix(alts).to_numpy(float) @ betas_ind.to_numpy().T   # utility: 55 options x 400 respondents
alt_records = alts.to_dict('records')

rows = []
for r in range(N_RESP):
    for t in range(N_TASKS):
        shown = list(rng_tasks.choice(len(profiles), N_ALTS, replace=False)) + [NONE_IDX]
        u = U_all[shown, r]                                         # deterministic utilities of the 4 options
        choice = int(np.argmax(u + rng_tasks.gumbel(size=len(shown))))   # + Gumbel noise -> logit choice
        for a, i in enumerate(shown):
            rows.append({'resp': r, 'task': t, 'alt': a, **alt_records[i], 'chosen': int(a == choice)})

cbc = pd.DataFrame(rows)   # long format: one row per alternative, 4 rows per task
print(cbc.shape, '| choices per task:', cbc.groupby(['resp', 'task'])['chosen'].sum().unique())
print(f"'None' chosen in {cbc.loc[cbc['catalog'] == 'NONE', 'chosen'].mean():.1%} of tasks")
cbc.head(8)

## 3. Estimate the aggregate MNL

In every task the respondent picks **one** of 4 options. The model for that choice is the **conditional (multinomial) logit**, the same formula as on the slide *Choice-Based Conjoint: The Gold Standard*:

$$P(j \mid \text{task}) = \frac{e^{U_j}}{\sum_k e^{U_k}}, \qquad U_j = \beta_{price}\,\text{price}_j + \beta_{catalog} + \beta_{ads} + \beta_{devices} \;(+\,\beta_{none} \text{ for 'None'})$$

The estimate is the β that makes the observed choices most likely (maximum likelihood). There is **no constant**: only differences in utility *within* a task matter, and the 'None' coefficient plays the role of the None option's intercept.

We write the likelihood ourselves in a few lines of numpy (`fit_mnl`). It is fast, and we need that speed for the bootstrap in step 7. The library version (`statsmodels` `ConditionalLogit`) gives the same coefficients (differences below 0.0002) but takes 10-20 s per fit; the cross-check is in the cell, commented out.

"Aggregate" means **one set of β for everybody**: the model ignores that people differ.

In [ ]:
def fit_mnl(X, chosen, n_alts, weights=None, start=None):
    # Conditional (multinomial) logit for CBC data.
    # Rows come in blocks of n_alts: one block = one choice task.
    # In each task P(j) = exp(U_j) / sum_k exp(U_k), with U = X @ beta.
    # weights: optional weight per task; start: starting values (both used by the bootstrap in step 7).
    Xt = np.asarray(X, float).reshape(-1, n_alts, X.shape[1])     # tasks x alternatives x features
    c  = np.asarray(chosen).reshape(-1, n_alts).argmax(axis=1)     # position of the chosen alternative
    w  = np.ones(len(Xt)) if weights is None else np.asarray(weights, float)
    task = np.arange(len(Xt))

    def probs(beta):
        u = Xt @ beta
        e = np.exp(u - u.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)

    def neg_loglik(beta):            # minus the log-likelihood and its gradient
        p = probs(beta)
        grad = Xt[task, c] - (p[..., None] * Xt).sum(axis=1)
        return -(w * np.log(p[task, c])).sum(), -(w[:, None] * grad).sum(axis=0)

    x0 = np.zeros(Xt.shape[2]) if start is None else np.asarray(start, float)
    res = minimize(neg_loglik, x0, jac=True, method='BFGS')
    # standard errors from the curvature of the log-likelihood (Hessian)
    p = probs(res.x)
    d = Xt - (p[..., None] * Xt).sum(axis=1, keepdims=True)
    D = d.reshape(-1, Xt.shape[2])
    H = (D * (w[:, None] * p).reshape(-1, 1)).T @ D     # sum over tasks of w * p * d d'
    se = np.sqrt(np.diag(np.linalg.inv(H)))
    return pd.Series(res.x, index=X.columns), pd.Series(se, index=X.columns)

# fit_mnl needs the rows in task blocks (alt 0, 1, 2, None): check it
assert (cbc['alt'].to_numpy().reshape(-1, N_ALTS + 1) == np.arange(N_ALTS + 1)).all(), 'rows not in task blocks'
X = design_matrix(cbc)
est, se = fit_mnl(X, cbc['chosen'], n_alts=N_ALTS + 1)
summary = pd.DataFrame({'coef': est, 'std err': se, 'z': est / se,
                        'p-value': 2 * norm.sf(np.abs(est / se))})
print(summary.round(4))

# Cross-check with the library (same likelihood; coefficients agree within 0.0002; 10-20 s):
# from statsmodels.discrete.conditional_models import ConditionalLogit
# print(ConditionalLogit(cbc['chosen'], X, groups=cbc['resp'] * 100 + cbc['task']).fit(disp=False).params)

In [ ]:
# compare estimated vs true
comp = pd.DataFrame({'true': pd.Series(TRUE), 'estimated': est}).round(3)
comp['gap'] = (comp['estimated'] - comp['true']).round(3)

CHECK_RECOVERY = True   # set to False for the exercises that break recovery on purpose
if CHECK_RECOVERY:
    assert abs(comp.loc['price', 'gap']) < 0.05, 'price coefficient not recovered'
    assert comp['gap'].abs().max() < 0.2, 'a part-worth is more than 0.2 utils off'
    print('Recovery check passed: price within 0.05, every part-worth within 0.2 of the truth.')
comp

**Read the gap column.** The price coefficient is recovered almost exactly (−0.248 vs −0.25). Catalog, ads and devices are within ±0.06 of the truth; 'None' is the furthest off (−1.84 vs −2.0).

Why are the gaps not zero? **Sampling noise**: 400 respondents × 8 tasks is a finite sample. Compare each gap with the `std err` column in the cell above: every gap is smaller than 1.5 standard errors, so noise alone explains them. The spread in tastes (sd 0.25) does little harm here; a large spread in *price sensitivity* does much more (mini-exercise 5).

Try `N_RESP = 50` (and `CHECK_RECOVERY = False`): the standard errors almost triple and the largest gap passes 0.25. Sample size matters.

## 4. Attribute importance & willingness-to-pay

**Importance** = range of part-worths within an attribute, normalised across all attributes (textbook conjoint convention). It depends on the levels you chose to test: a wider price range makes price look more important.

**WTP** for moving from level A to level B = (β_B − β_A) / (−β_price). Read it as 'how many extra EUR/month a customer would pay to get level B instead of A'.

Because we simulated the data, we can print the **true** values next to the estimates.

In [ ]:
def importance(beta):
    ranges = {
        'price':   abs(beta['price'] * (max(PRICE) - min(PRICE))),
        'catalog': max(0, beta['cat_Standard'], beta['cat_Premium']) - min(0, beta['cat_Standard'], beta['cat_Premium']),
        'ads':     abs(beta['ads_No']),
        'devices': max(0, beta['dev_2'], beta['dev_4']) - min(0, beta['dev_2'], beta['dev_4']),
    }
    total = sum(ranges.values())
    return pd.Series({k: 100 * v / total for k, v in ranges.items()})

b = est.to_dict()
imp = pd.DataFrame({'estimated': importance(b), 'true': importance(TRUE)}).round(1)
print('Attribute importance (%):'); print(imp)

fig, ax = plt.subplots(figsize=(6,3.2))
ax.barh(imp.index, imp['estimated'], color='#1B263B')
ax.set_xlabel('Importance (%)'); ax.set_title('Attribute importance (estimated)')
plt.tight_layout(); plt.show()

In [ ]:
# Willingness to pay
def wtp(beta):
    return pd.Series({
        'Standard vs Basic':  -beta['cat_Standard'] / beta['price'],
        'Premium vs Basic':   -beta['cat_Premium']  / beta['price'],
        'No ads vs With ads': -beta['ads_No']       / beta['price'],
        '2 devices vs 1':     -beta['dev_2']        / beta['price'],
        '4 devices vs 1':     -beta['dev_4']        / beta['price'],
    })

wtp_table = pd.DataFrame({'estimated': wtp(b), 'true': wtp(TRUE)})
wtp_table['gap %'] = 100 * (wtp_table['estimated'] / wtp_table['true'] - 1)
print(f"No-ads WTP: estimated EUR {wtp_table.loc['No ads vs With ads', 'estimated']:.2f}/month, "
      f"true EUR {wtp_table.loc['No ads vs With ads', 'true']:.2f}/month")
wtp_table.round({'estimated': 2, 'true': 2, 'gap %': 0}).rename_axis('WTP (EUR/month)')

**How to read:** an ad-free tier is worth about **€3.07 more per month** to the average customer (the true value we simulated is €3.20). Compare this to the price gap your CFO wants to charge — that is the negotiation.

All estimated WTPs are within about ±11% of the truth, because the price coefficient, the denominator of every WTP, is recovered well. If it were biased towards zero, every WTP would be inflated by the same factor (mini-exercise 5).

## 5. Market simulator — share of preference

Given a set of competing products (incl. 'None'), the MNL gives the **share of preference**:
$$\text{share}_i = \frac{e^{U_i}}{\sum_j e^{U_j}}$$

Define a competitive scenario:
- **Our product**: Premium / No ads / 4 devices, price varied **7.99 → 15.99** (the tested range)
- **Competitor A**: Standard / With ads / 2 devices @ 9.99
- **Competitor B**: Premium / With ads / 4 devices @ 13.99
- **None** (status quo)

Why stop at 15.99? Above the highest tested level we would only extend a straight line we never measured. This is the rule we stated with the slide *Connecting Market Share to Revenue and Pricing Decisions*: simulate revenue only **within the design range**.

In [ ]:
def scenario(our_price):
    # one row per option; our price is the only thing we change
    return pd.DataFrame([
        {'option': 'Us (Premium/No ads/4)', 'price': our_price, 'catalog': 'Premium',  'ads': 'No ads',   'devices': 4},
        {'option': 'Competitor A',          'price': 9.99,      'catalog': 'Standard', 'ads': 'With ads', 'devices': 2},
        {'option': 'Competitor B',          'price': 13.99,     'catalog': 'Premium',  'ads': 'With ads', 'devices': 4},
        {'option': 'None',                  'price': 0.0,       'catalog': 'NONE',     'ads': 'NONE',     'devices': 0},
    ]).set_index('option')

def logit_shares(df, beta):
    u = utility(df, beta)
    e = np.exp(u - u.max())
    return pd.Series(e / e.sum(), index=df.index)

# price grid = the tested range only, in steps of 5 cents
prices = np.round(np.arange(min(PRICE), max(PRICE) + 1e-9, 0.05), 2)

# all scenarios on the grid stacked once, so we can score any beta in one matrix product
X_GRID = design_matrix(pd.concat([scenario(p) for p in prices])).to_numpy(float)

def share_curves(beta):
    # same softmax as logit_shares, for every price on the grid at once (rows = prices)
    u = (X_GRID @ pd.Series(beta)[FEATURES].to_numpy(float)).reshape(len(prices), -1)
    e = np.exp(u - u.max(axis=1, keepdims=True))
    return pd.DataFrame(e / e.sum(axis=1, keepdims=True), index=prices, columns=scenario(0).index)

shares = share_curves(b)

fig, ax = plt.subplots(figsize=(7,4))
ax.stackplot(prices, shares.T.values, labels=shares.columns,
             colors=['#D4A017','#5DADE2','#58D68D','#BDC3C7'], alpha=0.85)
ax.set_xlim(min(PRICE), max(PRICE))
ax.set_xlabel('Our price (EUR/month), tested range only'); ax.set_ylabel('Share of preference')
ax.set_title('Market share vs our price')
ax.legend(loc='lower left', fontsize=8); plt.tight_layout(); plt.show()

print(shares.iloc[[0, len(prices) // 2, -1]].round(3))

### 5b. One scenario, three share rules

The simulator above uses **one** rule: logit shares at the aggregate estimates (a single "representative customer"). The slide *How Utilities Translate to Market Share Predictions* shows two more:
- **First-Choice (Max Utility)**: each respondent "buys" the option with the highest utility;
- **Logit, averaged over respondents**: each respondent splits the purchase by the softmax, then we average.

Both need **individual** part-worths. In a real study they come from Hierarchical Bayes (HB); here we use the individual β we simulated in step 2.

In [ ]:
P_TEST = 11.99
scen = scenario(P_TEST)
U_ind = betas_ind.to_numpy() @ design_matrix(scen).to_numpy(float).T    # 400 respondents x 4 options

first_choice = np.bincount(U_ind.argmax(axis=1), minlength=len(scen)) / N_RESP
e = np.exp(U_ind - U_ind.max(axis=1, keepdims=True))
logit_avg = (e / e.sum(axis=1, keepdims=True)).mean(axis=0)

rules = pd.DataFrame({'First-Choice': first_choice,
                      'Logit, averaged over respondents': logit_avg,
                      'Logit at aggregate MNL': logit_shares(scen, b)}, index=scen.index)
print(f'Share of preference (%) with our price at {P_TEST} EUR')
(100 * rules).round(1)

**Same utilities, very different answers.** First-Choice gives us **95.8%** of the market; both logit rules give **53-55%**. First-Choice is winner-takes-all: a respondent who likes us only slightly more counts as a full sale. The *Rocket R1* case shows the same pattern (84.4% under Max Utility vs 47.7% under Logit).

The averaged logit and the aggregate logit are close here because price sensitivity varies little between respondents. The share rule is a modelling decision: state it next to every share number you report.

## 6. Potential demand & revenue

Assume the addressable market is **M = 2,000,000 households** in our country. Potential demand at each price = share × M. Revenue = demand × price. This is the formula on the slide *Connecting Market Share to Revenue and Pricing Decisions*: Revenue = MKS × Price × Market.

Two things to read besides the optimum:
- the **shaded band**: every price where revenue is within 1% of the maximum. A flat top means the exact optimum matters less than it looks;
- the **price elasticity** at the optimum. At a revenue maximum it equals −1, the unit-elasticity rule on the slide *Optimal Pricing via Calculus*.

In [ ]:
M = 2_000_000
share_us = shares.iloc[:, 0].to_numpy()
demand   = share_us * M
revenue  = demand * prices

opt   = int(np.argmax(revenue))
p_opt = prices[opt]
flat  = prices[revenue >= 0.99 * revenue[opt]]               # prices within 1% of the best revenue
elasticity = b['price'] * p_opt * (1 - share_us[opt])        # own-price elasticity of a logit share

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11,3.8))
ax1.plot(prices, demand/1e3, color='#1B263B', lw=2)
ax1.set_xlabel('Price (EUR/month)'); ax1.set_ylabel('Potential demand (k households)')
ax1.set_title('Demand curve')

ax2.plot(prices, revenue/1e6, color='#D4A017', lw=2)
ax2.axvspan(flat.min(), flat.max(), color='#D4A017', alpha=0.15,
            label=f'within 1% of max: {flat.min():.2f}-{flat.max():.2f}')
ax2.axvline(p_opt, color='red', ls='--', alpha=0.6, label=f'Optimal: {p_opt:.2f} EUR')
ax2.set_ylim(0, revenue.max() / 1e6 * 1.15)
ax2.set_xlabel('Price (EUR/month)'); ax2.set_ylabel('Monthly revenue (EUR, M)')
ax2.set_title('Revenue curve'); ax2.legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

print(f'Revenue-maximising price:       {p_opt:.2f} EUR/month')
print(f'Prices within 1% of max:        {flat.min():.2f} - {flat.max():.2f} EUR/month')
print(f'Share of preference at opt:     {share_us[opt]:.1%}')
print(f'Potential demand at opt:        ~{demand[opt]/1e6:.2f}M households')
print(f'Monthly revenue at opt:         ~{revenue[opt]/1e6:.1f}M EUR')
print(f'Annualised revenue, upper bound: ~{12*revenue[opt]/1e6:.0f}M EUR (assumes share = paying subscribers, no churn)')
print(f'Price elasticity at opt:        {elasticity:.2f}')

**Business read-out.** The revenue-maximising price is **€10.69**, but revenue stays within 1% of the maximum anywhere between **€9.79 and €11.64**. Missing the optimum by up to 90 cents either way costs less than 1% of revenue, so other arguments (positioning, competitor reaction, margin) can pick the final price inside the band.

At the optimum the price elasticity is **−1.00**: the unit-elasticity rule of the pricing deep dive, found here by grid search instead of calculus.

The annualised figure (~€160M) is an **upper bound**: it treats share of preference as paying subscribers and ignores churn (see the caveats in section 9).

## 7. Sensitivity — how much does the answer depend on the sample?

Bootstrap: draw 400 respondents **with replacement**, refit the MNL, recompute the optimal price; repeat 200 times. The spread shows **sampling uncertainty**: how much the answer would move with another sample of 400 people from the same market.

What it does **not** show: errors in the model itself (for example, one set of β for everybody). Those stay the same in every replicate.

In [ ]:
def optimal_price(beta):
    rev = share_curves(beta).iloc[:, 0].to_numpy() * M * prices
    i = int(np.argmax(rev))
    return prices[i], rev[i]

N_BOOT = 200
rng_boot = np.random.default_rng(2026)
boot_prices, boot_revs = [], []
for _ in range(N_BOOT):
    draw = rng_boot.choice(N_RESP, size=N_RESP, replace=True)         # resample respondents
    w = np.repeat(np.bincount(draw, minlength=N_RESP), N_TASKS)       # how often each respondent's tasks count
    beta_b, _ = fit_mnl(X, cbc['chosen'], N_ALTS + 1, weights=w, start=est)
    bp, br = optimal_price(beta_b)
    boot_prices.append(bp); boot_revs.append(br)
boot_prices, boot_revs = np.array(boot_prices), np.array(boot_revs)

lo, hi = np.percentile(boot_prices, [5, 95])
print(f'Optimal price, 90% bootstrap interval: [{lo:.2f}, {hi:.2f}] EUR  ({N_BOOT} replicates)')
print(f'Max revenue,   90% bootstrap interval: [{np.percentile(boot_revs, 5)/1e6:.2f}, {np.percentile(boot_revs, 95)/1e6:.2f}] M EUR')
inside = np.mean((boot_prices >= flat.min()) & (boot_prices <= flat.max()))
print(f'Replicates whose optimum falls in the 1% band ({flat.min():.2f}-{flat.max():.2f}): {inside:.0%}')

# Only possible in a simulation: the revenue curve of the TRUE population (each respondent's own beta)
U_pop = (betas_ind.to_numpy() @ X_GRID.T).reshape(N_RESP, len(prices), -1)      # respondents x prices x options
e_pop = np.exp(U_pop - U_pop.max(axis=2, keepdims=True))
rev_pop = (e_pop / e_pop.sum(axis=2, keepdims=True))[:, :, 0].mean(axis=0) * M * prices
i_pop = int(np.argmax(rev_pop))
print(f'True-population optimum: {prices[i_pop]:.2f} EUR, {rev_pop[i_pop]/1e6:.2f} M EUR '
      f'(model estimate: {p_opt:.2f} EUR, {revenue[opt]/1e6:.2f} M EUR)')

fig, ax = plt.subplots(figsize=(6,3))
ax.axvspan(flat.min(), flat.max(), color='#D4A017', alpha=0.2, zorder=0, label='revenue within 1% of max')
ax.hist(boot_prices, bins=np.arange(boot_prices.min() - 0.125, boot_prices.max() + 0.25, 0.25),
        color='#1B263B', alpha=0.85, zorder=2)
ax.set_xlabel('Bootstrapped optimal price (EUR/month)'); ax.set_ylabel('replicates')
ax.set_title(f'Sampling uncertainty ({N_BOOT} bootstrap replicates)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Read-out.** Over 200 resamples the optimal price stays between **€10.44 and €11.04** (90% interval), and every replicate lands inside the 1% band (€9.79-11.64). Sampling noise does not threaten this decision.

Because we simulated the market, we can also do something impossible with real data: compute the revenue curve of the *true* population, respondent by respondent. Its optimum is **€10.59** and €13.0M a month, against the model's €10.69 and €13.3M. The model error is small here because price sensitivity varies little. The bootstrap cannot see this kind of error at all: run mini-exercise 5 and compare the two numbers again.

## 8. Mini-exercises

1. **Add a 4th level** to `Catalog` (e.g. `Premium+Sports`). Add it to `CATALOG`, add a column for it in `design_matrix` and its name in `FEATURES`, give it a true part-worth of +1.5 in `TRUE`, then *Run all*. Does the MNL recover it?
2. **Change the competitor mix**: drop Competitor B, add a free ad-supported tier (price=0, Basic, with ads, 1 device). What happens to the optimal price? Price 0 lies outside the tested range (7.99-15.99): how much do you trust the free tier's utility?
3. **Heterogeneous segments**: split respondents into 'students' (price coefficient −0.4) and 'families' (price −0.15). Re-fit the MNL globally vs per segment. Where does the aggregate model lie? (Hint: it is *not* the average.)
4. **Profit instead of revenue**: assume marginal cost = €3/month. In a **new cell** below step 6, compute `profit = demand * (prices - 3)` and find the price that maximises it. How far does the optimal price shift? (Leave the step-6 cell unchanged: step 7 reuses its revenue curve.)
5. **Why HB exists**: set `PRICE_SD = 0.25` in step 2 and `CHECK_RECOVERY = False` in step 3, then *Run all*. Compare the price coefficient, the 'None' coefficient and the WTP table with the truth. What happens to the WTP when the aggregate model meets people whose price sensitivity differs a lot?

## 9. Caveats — what this notebook does NOT do

- **No latent-class / mixed logit.** Real CBC analysis recovers heterogeneity with HB (Hierarchical Bayes) or LC-MNL. With strong heterogeneity, aggregate MNL shrinks the price coefficient towards zero and inflates WTP (exercise 5). Its shares also obey IIA: a new product takes share from every rival in the same proportion, however similar or different they are.
- **No purchase-frequency model.** Share of preference ≠ probability of purchase in a given month. A real revenue model needs choice × frequency × retention.
- **Static competitors.** Reality: competitors react. Equilibrium pricing is a game, not an optimisation.
- **Stated, not revealed, preference.** What people choose in a survey is not what they pay for. In deployment you would calibrate WTP against an A/B price test.
- **No efficient experimental design.** We use random profiles per task; a real CBC uses balanced/orthogonal or D-efficient designs.
- **The bootstrap measures sampling noise only**, not the errors of the model itself.

## Cross-references

- **Slides** (Module 9, conjoint): *Choice-Based Conjoint: The Gold Standard* (the MNL formula), *Five Steps from Survey to Business Decision*, *Experimental Design: Product Profiles*, *OLS Regression Estimates Utility for Each Feature Level* (part-worths and WTP), *How Utilities Translate to Market Share Predictions* (share rules), *Connecting Market Share to Revenue and Pricing Decisions* (revenue), the *Rocket R1* case study.
- **Next in the pricing deep dive**: *Marshall's Law (1890) — Still Fundamental*, *Optimal Pricing via Calculus* (elasticity −1 at the revenue optimum), *Accounting for Cost of Supply* (exercise 4: the profit optimum sits above the revenue optimum).
- **Related notebook**: `03_demand_eda_multi_product.ipynb` — same underlying logic when SKUs cannibalise.